# HemoLens — Local Prediction

Run hemoglobin estimation on your machine without Expo or Render.

**Setup (run once in terminal):**
```bash
cd backend
pip install -r requirements.txt ipywidgets matplotlib jupyter
jupyter notebook ../notebooks/hemolens_local.ipynb
```

Upload eye, nail, and/or palm images below and click **Run prediction**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from predict_local import load_models, predict_from_images, load_rgb_from_bytes

status = load_models()
print("Models ready")
print(f"  Eye (Ridge): {'yes' if status['eye_loaded'] else 'no'}")
print(f"  Multimodal: {'yes' if status['multimodal_loaded'] else 'no'}")
if status["multimodal_loaded"]:
    cfg = status["multimodal_config"]
    print(f"  Multimodal R²={cfg.get('r2')}  MAE={cfg.get('mae')} g/dL")

In [ ]:
def show_images(images):
    count = len(images)
    if count == 0:
        return
    fig, axes = plt.subplots(1, count, figsize=(4 * count, 4))
    if count == 1:
        axes = [axes]
    for ax, (label, rgb) in zip(axes, images.items()):
        ax.imshow(rgb)
        ax.set_title(label.capitalize())
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_result(result):
    print()
    if result.get("status") == "invalid_image":
        print("INVALID IMAGE")
        print(result.get("message", ""))
        for key, info in (result.get("validation") or {}).items():
            print(f"  {key}: score={info['score']:.2f} — {info['message']}")
        return

    print("SUCCESS")
    print(f"Hemoglobin: {result['hemoglobin_estimate']} {result['unit']}")
    print(f"Status: {result['health_status']}")
    print(result['health_message'])
    print(f"Model: {result.get('model')}")
    if result.get("modalities_used"):
        print(f"Modalities: {', '.join(result['modalities_used'])}")
    if result.get("eye_quality_score") is not None:
        print(f"Eye quality: {result['eye_quality_score']:.2f}")
    if result.get("warning"):
        print(f"Note: {result['warning']}")


def decode_upload(upload_widget):
    if not upload_widget.value:
        return None
    name, meta = next(iter(upload_widget.value.items()))
    return load_rgb_from_bytes(meta["content"]), name

In [ ]:
eye_upload = widgets.FileUpload(accept="image/*", multiple=False, description="Eye")
nail_upload = widgets.FileUpload(accept="image/*", multiple=False, description="Nail")
palm_upload = widgets.FileUpload(accept="image/*", multiple=False, description="Palm")
ridge_only = widgets.Checkbox(value=False, description="Use eye-only Ridge model")
run_btn = widgets.Button(description="Run prediction", button_style="primary")
output = widgets.Output()

upload_row = widgets.HBox([eye_upload, nail_upload, palm_upload])
controls = widgets.VBox([
    widgets.HTML("<b>Upload at least one image</b>"),
    upload_row,
    ridge_only,
    run_btn,
    output,
])


def on_run(_):
    with output:
        clear_output(wait=True)
        try:
            eye_data = decode_upload(eye_upload)
            nail_data = decode_upload(nail_upload)
            palm_data = decode_upload(palm_upload)

            eye = eye_data[0] if eye_data else None
            nail = nail_data[0] if nail_data else None
            palm = palm_data[0] if palm_data else None

            if not any([eye is not None, nail is not None, palm is not None]):
                print("Upload at least one image (eye, nail, or palm).")
                return

            preview = {}
            if eye is not None:
                preview["eye"] = eye
            if nail is not None:
                preview["nail"] = nail
            if palm is not None:
                preview["palm"] = palm
            show_images(preview)

            result = predict_from_images(
                eye=eye,
                nail=nail,
                palm=palm,
                prefer_multimodal=not ridge_only.value,
            )
            show_result(result)
        except Exception as exc:
            print(f"Error: {exc}")


run_btn.on_click(on_run)
display(controls)

## Alternative: predict from file paths

If you already have image files on disk, set the paths below and run this cell.

In [ ]:
from predict_local import load_rgb_from_path

EYE_PATH = ""   # e.g. "/Users/you/eye.jpg"
NAIL_PATH = ""  # optional
PALM_PATH = ""  # optional

paths = {"eye": EYE_PATH, "nail": NAIL_PATH, "palm": PALM_PATH}
images = {k: load_rgb_from_path(v) for k, v in paths.items() if v}

if not images:
    print("Set at least one path above (EYE_PATH, NAIL_PATH, or PALM_PATH).")
else:
    show_images(images)
    result = predict_from_images(
        eye=images.get("eye"),
        nail=images.get("nail"),
        palm=images.get("palm"),
    )
    show_result(result)